# 09 · FastAPI and CLI

Exercise the production-shaped boundary locally through TestClient, including typed input validation, metrics lookup, skills, and churn prediction.

**How to use this notebook**

1. Run cells from top to bottom.
2. Start with the CI profile; switch to `portfolio` only after the workflow is clear.
3. Change one parameter at a time and record the effect rather than chasing a high score.

> All records are deterministic synthetic data. Results are educational evidence, not a
> production or financial decision system.

In [ ]:
# Shared local/Colab bootstrap. Run this cell first.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "codex/colab-notebook-first",
                "--single-branch",
                "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git",
                str(repo),
            ],
            check=True,
        )
    os.chdir(repo)
    %pip install -q -e ".[full,dev]"
else:
    here = Path.cwd().resolve()
    if here.name == "notebooks":
        os.chdir(here.parent)

ROOT = Path.cwd().resolve()
assert (ROOT / "pyproject.toml").exists(), "Please open the repository root or notebooks folder."
print(f"environment={'Colab' if IN_COLAB else 'local/VS Code'}")
print(f"repository={ROOT}")

In [ ]:
import subprocess
import sys
from pathlib import Path

if not Path("artifacts/models/best_classical.joblib").exists():
    subprocess.run(
        [
            sys.executable,
            "scripts/run_classical.py",
            "--data-dir",
            "data/generated/ci",
            "--profile",
            "ci",
            "--output",
            "artifacts/metrics/ci/classical.json",
        ],
        check=True,
    )

In [ ]:
from fastapi.testclient import TestClient

from rookie_lab.serving.api import app

client = TestClient(app)
print(client.get("/health").json())
print("skills", client.get("/skills").json()["active_nodes"])
payload = {
    "age": 36,
    "tenure_days": 800,
    "region": "north",
    "channel": "web",
    "income": 72000,
    "visits_30d": 5,
    "support_tickets": 1,
    "loyalty_score": 0.72,
}
response = client.post("/predict/churn", json=payload)
response.status_code, response.json()

In [ ]:
invalid = {**payload, "age": 8}
response = client.post("/predict/churn", json=invalid)
print("Validation status", response.status_code)
display(response.json())

## Run as a service

In a VS Code terminal: `ROOKIE_LAB_DATA_DIR=data/generated/portfolio rookie-lab serve`, then open `/docs`. Colab notebooks should use TestClient rather than exposing a public tunnel.